In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import numpy as np
import scipy.linalg as la
import warnings
from pathlib import Path
from multiprocessing import Pool, cpu_count
from scipy.optimize import curve_fit

warnings.filterwarnings("ignore")

# =============================
# Model + MF machinery (mostly your original code, with light cleanups)
# =============================

class System:
    def __init__(self, T, L, t, mu, W, delta, U, V,
                 disorder, disorder1, disorder_gamma, disorder_eta,
                 npf, pbc, trs):
        # Build convenience arrays
        arr = np.ones(L)
        arr[0] = arr[-1] = pbc

        a1 = 1j * np.block([npf[L-1] * pbc, npf[0:L-1]])
        a2 = 1j * np.block([npf[1:L], npf[0] * pbc])
        d  = 1j * npf[3*L:4*L] * arr
        b  = 1j * npf[1*L:2*L]
        b1 = 1j * np.block([npf[2*L-1] * pbc, npf[1*L:2*L-1]])
        b2 = 1j * np.block([npf[1*L+1:2*L], npf[1*L] * pbc])
        c  = 1j * npf[2*L:3*L]

        i_gamma_j_gamma_jplus1 = 1j * npf[4*L:5*L]
        i_eta_j_eta_jplus1     = 1j * npf[5*L:6*L]

        e1 = np.block([i_eta_j_eta_jplus1[-1], i_eta_j_eta_jplus1[0:L-1]])
        e2 = np.block([i_eta_j_eta_jplus1[-2], i_eta_j_eta_jplus1[-1], i_eta_j_eta_jplus1[0:L-2]])
        f1 = np.block([i_gamma_j_gamma_jplus1[-1], i_gamma_j_gamma_jplus1[0:L-1]])
        f2 = i_gamma_j_gamma_jplus1

        H_mu = -1j * (-W * disorder + mu) * np.eye(L, k=0) - U * (a1 + a2) * np.eye(L, k=0) - V * d * np.eye(L, k=0)
        H_t  = 1j * (-t + delta) * np.eye(L, k=1) - U * b * np.eye(L, k=1)
        H_t[L-1, 0] = (1j * (-t + delta) - U * b[0]) * pbc
        H_t1 = 1j * (-t - delta - W * disorder1) * np.eye(L, k=-1) + U * c * np.eye(L, k=-1) + V * (b1 + b2) * np.eye(L, k=-1)
        H_t1[0, L-1] = (1j * (-t - delta - W * disorder1[0]) + U * c[L-1] + V * (b1[L-1] + b2[L-1])) * pbc
        H_t2 = V * a2 * np.eye(L, k=-2)
        H_t2[0, L-2] = V * a2[L-2] * pbc
        H_t2[1, L-1] = V * a2[L-1] * pbc
        H_maj_mf = H_mu + H_t + H_t1 + H_t2
        zero = 0 * np.eye(L, k=0)

        # TRS-breaking (gamma/eta) blocks; W_ratio and trs are globals
        H_gamma0 = trs * np.eye(L, k=1) * (-U * e1 - V * e2) + 1j * np.eye(L, k=1) * W * W_ratio * disorder_gamma
        H_eta0   = trs * np.eye(L, k=1) * (-U * f1 - V * f2) + 1j * np.eye(L, k=1) * W * W_ratio * disorder_eta
        H_gamma0[L-1, 0] = (trs * (-U * e1[0] - V * e2[0]) + 1j * W * W_ratio * disorder_gamma[0]) * pbc
        H_eta0[L-1, 0]   = (trs * (-U * f1[0] - V * f2[0]) + 1j * W * W_ratio * disorder_eta[0]) * pbc

        H_gamma = H_gamma0 - H_gamma0.T
        H_eta   = H_eta0 - H_eta0.T

        H = np.block([[ H_gamma, H_maj_mf],
                      [H_maj_mf.conj().T,   H_eta]])
        self.H = H

        # Ground-state correlator (no dissipation)
        eigvalsU, eigvecsU = np.linalg.eigh(self.H)
        self.gamma_eta = (np.conj(eigvecsU)).dot(np.diag(np.heaviside(-eigvalsU, 0.5))).dot(eigvecsU.T)

        majorana_to_c = np.sqrt(0.5) * np.block([[np.eye(L),  np.eye(L)],
                                                 [1j*np.eye(L), -1j*np.eye(L)]])
        self.nij = (np.conj(majorana_to_c.T)).dot(self.gamma_eta).dot(majorana_to_c)

        self.L = L
        self.E = np.trace(np.diag(eigvalsU * np.heaviside(-eigvalsU, 0.5)))

    def calculation_npf(self):
        L = self.L
        gamma_eta = (self.gamma_eta)[0:L, L:2*L]

        ai = np.zeros(L); bi = np.zeros(L); ci = np.zeros(L); di = np.zeros(L)
        xi = np.zeros(L); yi = np.zeros(L)

        for i in range(L):
            ai[i] = np.real((-1j) * gamma_eta[i, i])
            bi[i] = np.real((1j) * gamma_eta[(i+1) % L, i])
            ci[i] = np.real((-1j) * gamma_eta[i, (i+1) % L])
            di[i] = np.real((1j) * gamma_eta[(i+1) % L, (i-1) % L])

        gamma_gamma = (self.gamma_eta)[0:L, 0:L]
        eta_eta    = (self.gamma_eta)[L:2*L, L:2*L]
        for i in range(L):
            xi[i] = np.real((-1j) * gamma_gamma[i, (i+1) % L])
            yi[i] = np.real((-1j) * eta_eta[i, (i+1) % L])

        return np.block([ai, bi, ci, di, xi, yi])

    def ground_state_energy(self):
        return self.E

    def index_npf(self):
        L = self.L
        newH = self.H
        w, v = np.linalg.eigh(newH)
        nij = self.nij

        z1 = 0j; z2 = 0j
        central_chargeXXX = 0
        sinLxXXX = 0
        S_ent_lxXXX = 0
        CDW_component = 0
        zz = 0

        return (
            np.heaviside(1 - np.abs(z1+0.0j), 0.5) + np.heaviside(1 - np.abs(z2+0.0j), 0.5),
            None,
            (w, v), central_chargeXXX, sinLxXXX, S_ent_lxXXX, CDW_component, zz, nij, self.E
        )

# =============================
# Helper for central charge fit
# =============================

def linear_extrapol(x, c, s0):
    return c * x + s0

# =============================
# NEW: CDW, spectrum, and curvature helpers
# =============================

def cdw_q_from_ni(n_i):
    """
    CDW component from site densities n_i:
        q = 2 * sum_{k>0} |n(k)|^4 / ( sum_{k>0} |n(k)|^2 )^2
    We remove the mean; rfft half-spectrum; normalization cancels.
    """
    n = np.asarray(n_i, dtype=float)
    n = n - n.mean()
    n_k = np.fft.rfft(n)
    p2 = np.abs(n_k[1:])**2
    s2 = p2.sum()
    if s2 == 0:
        return 0.0
    s4 = (p2**2).sum()
    return float(2.0 * s4 / (s2**2))


def density_spectrum(n_i):
    """
    Return (k_grid, |n_k|) for mean-removed densities.
    k_grid in radians/site on [0, pi] for rfft (L even) or up to Nyquist for L odd.
    """
    n = np.asarray(n_i, dtype=float)
    n = n - n.mean()
    n_k = np.fft.rfft(n)
    k_grid = 2.0 * np.pi * np.fft.rfftfreq(len(n), d=1.0)  # rad/site
    Nk_env = np.abs(n_k)                                   # envelope = |n_k|
    return k_grid, Nk_env


def _second_derivative_boundary_safe(y, i, dk):
    """
    Boundary-safe finite-difference estimate of f'' at index i on a uniform grid.
    Prefers central 3-point; otherwise uses one-sided 4-point formulas.
    Returns NaN if insufficient points exist.
    """
    N = len(y)
    if N < 3:
        return np.nan
    if (i - 1) >= 0 and (i + 1) < N:
        return (y[i - 1] - 2.0 * y[i] + y[i + 1]) / (dk ** 2)
    if (i + 3) < N:
        return (2.0 * y[i] - 5.0 * y[i + 1] + 4.0 * y[i + 2] - y[i + 3]) / (dk ** 2)
    if (i - 3) >= 0:
        return (2.0 * y[i] - 5.0 * y[i - 1] + 4.0 * y[i - 2] - y[i - 3]) / (dk ** 2)
    return np.nan


def curvature_at_peak_extremum(k_grid, Nk_env, deg=3, m_points=11, prefer="symmetric"):
    """
    Curvature near the dominant peak with robust fallbacks.

    Returns (curv, q_idx, Q_data, Q_fit):
      - curv  : curvature value used
      - q_idx : index of the data-peak (ignoring DC)
      - Q_data: k at the data-peak bin center
      - Q_fit : k at the fitted extremum (if found), else Q_data

    Strategy:
      1) Identify the data peak Q_data (argmax over k>0).
      2) Fit a polynomial p(x) on a local window around Q_data, x=(k-Q_data)/dk.
      3) If a stationary point x* exists in-window, report curvature at that extremum: p''(x*)/dk^2.
      4) If no extremum is found, evaluate curvature at Q_data via the fit: p''(0)/dk^2.
      5) If the fit fails, fall back to boundary-safe finite-difference at Q_data.
    """
    k_grid = np.asarray(k_grid, float)
    y = np.asarray(Nk_env, float)

    if len(k_grid) != len(y) or len(k_grid) < 3:
        return np.nan, -1, np.nan, np.nan
    if np.all(y[1:] == 0.0):
        return np.nan, -1, np.nan, np.nan

    q_idx = 1 + int(np.argmax(y[1:]))
    Q_data = float(k_grid[q_idx])
    dk = float(k_grid[1] - k_grid[0]) if len(k_grid) > 1 else 1.0

    m_points = max(m_points, deg + 1)
    if prefer == "symmetric":
        half = m_points // 2
        start = max(1, q_idx - half)
        stop  = min(len(k_grid), q_idx + half + 1)
        if (stop - start) < (deg + 1) or not (start <= q_idx < stop):
            start = max(1, q_idx - (m_points - 1))
            stop  = q_idx + 1
    else:
        start = max(1, q_idx - (m_points - 1))
        stop  = q_idx + 1

    idx = np.arange(start, stop)
    if len(idx) < 3:
        curv_fd = _second_derivative_boundary_safe(y, q_idx, dk)
        return float(curv_fd) if np.isfinite(curv_fd) else np.nan, q_idx, Q_data, Q_data

    cur_deg = min(deg, len(idx) - 1)
    if cur_deg < 2:
        curv_fd = _second_derivative_boundary_safe(y, q_idx, dk)
        return float(curv_fd) if np.isfinite(curv_fd) else np.nan, q_idx, Q_data, Q_data

    x = (k_grid[idx] - Q_data) / dk
    yy = y[idx]
    try:
        coeffs = np.polyfit(x, yy, cur_deg)
        p = np.poly1d(coeffs)
        dp = p.deriv(1)
        roots = dp.r
        x_min, x_max = float(x[0]), float(x[-1])
        real_roots = [r.real for r in roots if abs(getattr(r, "imag", 0.0)) < 1e-12]
        in_window = [xr for xr in real_roots if (x_min - 1e-12) <= xr <= (x_max + 1e-12)]

        if len(in_window) > 0:
            x_star = min(in_window, key=lambda xr: abs(xr))
            curv = float(p.deriv(2)(x_star)) / (dk ** 2)
            Q_fit = float(Q_data + x_star * dk)
            return curv, q_idx, Q_data, Q_fit
        else:
            curv_at_Q = float(p.deriv(2)(0.0)) / (dk ** 2)
            return curv_at_Q, q_idx, Q_data, Q_data
    except Exception:
        curv_fd = _second_derivative_boundary_safe(y, q_idx, dk)
        return float(curv_fd) if np.isfinite(curv_fd) else np.nan, q_idx, Q_data, Q_data

# =============================
# One-realization worker (runs MF, returns single-realization observables)
# =============================

def run_one_realization(task):
    (
        i_w, i_g, WX, g_val, seed,
        L, t0, delta0, mu0, VY_to_UY_ratio, T, pbc, trs,
        delta_mf, fric, mf_precision, jj_iter, Rq, lx_set, sinLx, W_ratio_param
    ) = task

    globals()['W_ratio'] = W_ratio_param

    rng = np.random.default_rng(seed)
    UY = 2 * g_val
    VY = VY_to_UY_ratio * UY

    dis        = rng.uniform(-1.0, 1.0, size=L)
    dis1       = rng.uniform(-1.0, 1.0, size=L)
    dis_gamma  = rng.uniform(-1.0, 1.0, size=L)
    dis_eta    = rng.uniform(-1.0, 1.0, size=L)

    npf = delta_mf * rng.uniform(-1.0, 1.0, size=6 * L)

    try:
        sys0 = System(T=T, L=L, t=t0, mu=mu0, W=WX, delta=delta0, U=0, V=0,
                      disorder=dis, disorder1=dis1, disorder_gamma=dis_gamma, disorder_eta=dis_eta,
                      npf=npf, pbc=pbc, trs=trs)
        npf = sys0.calculation_npf()
    except Exception:
        pass

    mf_converged = False
    jj = 0
    npf_y = npf
    energy_GS = 0.0
    while jj < jj_iter:
        #npf_x = npf_y
        sys = System(T=T, L=L, t=t0, mu=mu0, W=WX, delta=delta0, U=UY, V=VY,
                     disorder=dis, disorder1=dis1, disorder_gamma=dis_gamma, disorder_eta=dis_eta,
                     npf=npf, pbc=pbc, trs=trs)
        npf_y = sys.calculation_npf()
        #eps = float(np.max(np.abs(npf_x - npf_y)))
        eps = float(np.max(np.abs(npf - npf_y)))
        if eps < mf_precision:
            mf_converged = True
            energy_GS = float(sys.ground_state_energy())
            break
        #npf = (1 - fric) * npf_y + fric * npf_x
        npf = (1 - fric) * npf_y + fric * npf
        jj += 1

    sys = System(T=T, L=L, t=t0, mu=mu0, W=WX, delta=delta0, U=UY, V=VY,
                 disorder=dis, disorder1=dis1, disorder_gamma=dis_gamma, disorder_eta=dis_eta,
                 npf=npf, pbc=pbc, trs=trs)
    results = sys.index_npf()
    nij = results[8]
    eig_vals = results[2][0]

    delta_e = eig_vals[1:2*L] - eig_vals[0:2*L-1]
    r = np.minimum(delta_e[:-1], delta_e[1:]) / np.maximum(delta_e[:-1], delta_e[1:])
    AGR = float(np.mean(r))

    psi = results[2][1][:, L]
    ipr = float(np.sum(np.abs(psi)**4))

    n_i = np.real(np.diag(nij)[:L])
    cdw_q = cdw_q_from_ni(n_i)

    k_grid_local, Nk_env = density_spectrum(n_i)

    S_ent_lx = []
    for lx in lx_set:
        q = 0
        S_ent = 0.0
        while q <= (L - lx):
            qx = Rq * q
            if qx > L - lx - 1:
                break
            Cij = np.block([
                [nij[qx:qx+lx, qx:qx+lx],           nij[qx:qx+lx, qx+L:qx+L+lx]],
                [nij[qx+L:qx+L+lx, qx:qx+lx],       nij[qx+L:qx+L+lx, qx+L:qx+L+lx]]
            ])
            Lambda, _ = la.eig(Cij)
            mask = (np.abs(Lambda) > 0) & (np.abs(Lambda) < 1)
            if np.any(mask):
                S_ent = (q * S_ent - np.sum(Lambda[mask] * np.log(Lambda[mask]))) / (q + 1)
            q += 1
        S_ent_lx.append(np.real(S_ent))

    S_ent_lx = np.array(S_ent_lx, dtype=float)
    return (i_w, i_g, S_ent_lx, AGR, ipr, cdw_q, mf_converged, jj, Nk_env)

# =============================
# Orchestrator: parallel over (W, g), sequential over realizations with early stopping
# =============================

def run_adaptive_phase_diagram(
    # Physics / model params
    L=250, t0=0.5, delta0=0.5, int_stagg=1, VY_to_UY_ratio=1.0,
    T=1.0, pbc=1, trs=0, W_ratio=0.0,
    # Grid
    g_values=np.linspace(-0.5, 0.5, 41),
    WX_range=np.asarray([i**2/100 for i in range(11)]),
    # MF loop
    delta_mf=1e-3, fric=0.5, mf_precision=1e-9, jj_iter=100,
    # Entropy fit settings
    lx0=10, lx1=40, dlx=5, Rq=1,
    # Realizations / stopping
    dis_number_max=1000, central_tol=1e-2, max_consecutive_fail=10,
    min_realizations_before_stop=200,
    se_tol=3e-3,
    # Runtime
    processes=None,
    # IO
    out_dir="main_fig_BDI_adaptive/",
    save_prefix="adaptive_majoranas"
):
    """
    Adaptive averaging across disorder realizations:
      * Running mean of S(lx); fit c from mean S vs sinLx each step.
      * Running mean of |n_k|; compute curvature near peak with robust fallbacks.
      * Also compute Q_avg = sum_{k>0} k * <|n_k|> from the averaged envelope (unnormalized).
      * Early-stop on Δc̄ and SE(c) after a minimum sample count.
    """
    mu0 = -int_stagg * (t0 + delta0)
    lx_set = np.arange(lx0, lx1 + dlx, dlx)
    sinLx = [ (1.0/3.0) * np.log( 2*(2*L)/np.pi * np.sin(np.pi*(2*lx)/(2*L)) ) for lx in lx_set ]
    sinLx = np.array(sinLx, dtype=float)

    WN = len(WX_range); GN = len(g_values); LN = len(lx_set)

    # Spectrum grid
    Lk = L // 2 + 1
    k_grid = 2.0 * np.pi * np.fft.rfftfreq(L, d=1.0)  # rad/site

    # State arrays
    counts   = np.zeros((WN, GN), dtype=int)
    c_avg    = np.full((WN, GN), np.nan, dtype=float)
    s0_avg   = np.full((WN, GN), np.nan, dtype=float)
    agr_avg  = np.full((WN, GN), np.nan, dtype=float)
    ipr_avg  = np.full((WN, GN), np.nan, dtype=float)
    last_c   = np.full((WN, GN), np.inf, dtype=float)
    cdw_avg  = np.full((WN, GN), np.nan, dtype=float)
    active   = np.ones((WN, GN), dtype=bool)
    fail_streak = np.zeros((WN, GN), dtype=int)

    S_mean = np.full((WN, GN, LN), np.nan, dtype=float)

    Nk_env_mean = np.full((WN, GN, Lk), np.nan, dtype=float)
    curv_at_Q   = np.full((WN, GN), np.nan, dtype=float)
    Q_idx_arr   = np.full((WN, GN), -1, dtype=int)
    Q_val_arr   = np.full((WN, GN), np.nan, dtype=float)
    Q_ext_arr   = np.full((WN, GN), np.nan, dtype=float)

    # NEW: Q_avg = sum_{k>0} k * <|n_k|>
    Q_avg_arr   = np.full((WN, GN), np.nan, dtype=float)

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    if processes is None:
        processes = max(1, cpu_count())

    print(f"Parallel over grid: |W|={WN}, |g|={GN}, processes={processes}")
    print(f"Up to {dis_number_max} realizations; early stop when Δc̄ < {central_tol} or SE(c) < {se_tol} (after {min_realizations_before_stop}).")

    seed0 = 123456789
    for n in range(dis_number_max):
        tasks = []
        for i_w, WX in enumerate(WX_range):
            for i_g, g_val in enumerate(g_values):
                if not active[i_w, i_g]:
                    continue
                seed = hash((seed0, i_w, i_g, n)) & 0xFFFFFFFF
                tasks.append((i_w, i_g, WX, g_val, seed,
                              L, t0, delta0, mu0, VY_to_UY_ratio, T, pbc, trs,
                              delta_mf, fric, mf_precision, jj_iter, Rq, lx_set, sinLx, W_ratio))

        if not tasks:
            print("All grid points converged or excluded. Stopping.")
            break

        print(f"Realization {n+1}/{dis_number_max} — active points: {len(tasks)}")

        with Pool(processes=processes) as pool:
            results = pool.map(run_one_realization, tasks)

        for res in results:
            i_w, i_g, S_ent_lx_single, agr_single, ipr_single, cdw_single, mf_ok, jj, Nk_env_single = res

            if not mf_ok:
                fail_streak[i_w, i_g] += 1
                if fail_streak[i_w, i_g] >= max_consecutive_fail:
                    active[i_w, i_g] = False
                continue
            else:
                fail_streak[i_w, i_g] = 0

            kcnt = counts[i_w, i_g]
            if kcnt == 0 or np.any(np.isnan(S_mean[i_w, i_g])):
                S_mean[i_w, i_g] = S_ent_lx_single
            else:
                S_mean[i_w, i_g] = S_mean[i_w, i_g] + (S_ent_lx_single - S_mean[i_w, i_g]) / (kcnt + 1)

            if kcnt == 0 or np.any(np.isnan(Nk_env_mean[i_w, i_g])):
                Nk_env_mean[i_w, i_g] = Nk_env_single
            else:
                Nk_env_mean[i_w, i_g] += (Nk_env_single - Nk_env_mean[i_w, i_g]) / (kcnt + 1)

            # --- NEW: Q_avg = sum_{k>0} k * <|n_k|> (using current averaged envelope)
            Y = Nk_env_mean[i_w, i_g]
            Q_avg_val = Y[-1]/L #np.sqrt(float(np.nansum(k_grid[1:]**2 * Y[1:]))/float(np.nansum( Y[1:])))
            Q_avg_arr[i_w, i_g] = Q_avg_val

            if kcnt == 0:
                agr_avg[i_w, i_g] = agr_single
                ipr_avg[i_w, i_g] = ipr_single
                cdw_avg[i_w, i_g] = cdw_single
            else:
                agr_avg[i_w, i_g] += (agr_single - agr_avg[i_w, i_g]) / (kcnt + 1)
                ipr_avg[i_w, i_g] += (ipr_single - ipr_avg[i_w, i_g]) / (kcnt + 1)
                cdw_avg[i_w, i_g] += (cdw_single - cdw_avg[i_w, i_g]) / (kcnt + 1)

            counts[i_w, i_g] = kcnt + 1

            y = S_mean[i_w, i_g]
            mask = np.isfinite(y) & np.isfinite(sinLx)
            c_fit = np.nan; s0_fit = np.nan; c_se = np.inf
            try:
                if np.count_nonzero(mask) >= 2:
                    popt, pcov = curve_fit(linear_extrapol, sinLx[mask], y[mask])
                    c_fit, s0_fit = float(popt[0]), float(popt[1])
                    if (pcov is not None) and (pcov.shape == (2, 2)) and np.isfinite(pcov[0, 0]) and (pcov[0, 0] >= 0):
                        c_se = float(np.sqrt(pcov[0, 0]))
                    else:
                        c_se = np.inf
            except Exception:
                try:
                    p = np.polyfit(sinLx[mask], y[mask], 1)
                    c_fit, s0_fit = float(p[0]), float(p[1])
                    c_se = np.inf
                except Exception:
                    c_fit, s0_fit, c_se = np.nan, np.nan, np.inf

            c_avg[i_w, i_g] = c_fit
            s0_avg[i_w, i_g] = s0_fit

            prev = last_c[i_w, i_g]
            diff = np.inf if (not np.isfinite(prev) or not np.isfinite(c_fit)) else abs(c_fit - prev)
            eligible = counts[i_w, i_g] >= min_realizations_before_stop
            stop_by_delta = eligible and np.isfinite(c_fit) and (diff < central_tol)
            stop_by_se    = eligible and np.isfinite(c_fit) and (c_se < se_tol)
            if stop_by_delta or stop_by_se:
                active[i_w, i_g] = False
            last_c[i_w, i_g] = c_fit

            try:
                curv, q_idx, Q_data, Q_fit = curvature_at_peak_extremum(
                    k_grid, Nk_env_mean[i_w, i_g], deg=3, m_points=11, prefer="symmetric"
                )
            except Exception:
                curv, q_idx, Q_data, Q_fit = np.nan, -1, np.nan, np.nan

            curv_at_Q[i_w, i_g] = curv
            Q_idx_arr[i_w, i_g] = q_idx
            Q_val_arr[i_w, i_g] = Q_data
            Q_ext_arr[i_w, i_g] = Q_fit

        # Save after each realization
        np.save(out_path / f"{2*L}_majoranas_c.npy", c_avg)
        np.save(out_path / f"{2*L}_majoranas_agr.npy", agr_avg)
        np.save(out_path / f"{2*L}_majoranas_ipr.npy", ipr_avg)
        np.save(out_path / f"{2*L}_majoranas_counts.npy", counts)
        np.save(out_path / f"{2*L}_majoranas_active.npy", active)
        np.save(out_path / f"{2*L}_majoranas_cdw.npy", cdw_avg)

        # Spectrum + curvature + Q_avg artifacts
        np.save(out_path / f"{2*L}_nk_env_mean.npy", Nk_env_mean)
        np.save(out_path / f"{2*L}_nk_curvature.npy", curv_at_Q)
        np.save(out_path / f"{2*L}_nk_Q_idx.npy", Q_idx_arr)
        np.save(out_path / f"{2*L}_nk_Q_value.npy", Q_val_arr)
        np.save(out_path / f"{2*L}_nk_Qext_value.npy", Q_ext_arr)
        np.save(out_path / f"{2*L}_nk_Qavg.npy", Q_avg_arr)

    print("Done.")
    return (
        c_avg, agr_avg, ipr_avg, cdw_avg,
        counts, active,
        Nk_env_mean, curv_at_Q, Q_val_arr, Q_ext_arr,
        Q_avg_arr
    )

if __name__ == "__main__":

    # -------------------------------------------------------------------------
    # Symmetry-class and disorder settings
    # -------------------------------------------------------------------------

    # Relative strength of the time-reversal-symmetry-breaking disorder.
    #
    # In addition to the BDI-type disorder terms, the class-D Hamiltonian
    # contains random hopping between Majoranas of the same type:
    #
    #     i * W * W_ratio * disorder_gamma
    #     i * W * W_ratio * disorder_eta
    #
    # W_ratio = 1.0:
    #     the gamma-gamma and eta-eta disorder amplitudes have the same overall
    #     scale W as the BDI-type disorder.
    #
    # W_ratio = 0.0:
    #     the explicit TRS-breaking random hopping is absent.
    #
    # Therefore, W_ratio > 0 is the essential setting that places the
    # disordered model in symmetry class D.
    W_ratio = 1.0

    # Switch controlling the interaction-generated mean-field terms in the
    # gamma-gamma and eta-eta blocks:
    #
    #     H_gamma ~ trs * (-U*e1 - V*e2)
    #     H_eta   ~ trs * (-U*f1 - V*f2)
    #
    # trs = 1:
    #     retain these same-Majorana interaction-generated mean fields.
    #
    # trs = 0:
    #     suppress only these interaction-generated contributions.
    #
    # Important: trs does not suppress the random class-D disorder terms.
    # Those terms are controlled separately by W_ratio.
    trs = 1

    (
        c, agr, ipr, cdw,
        counts, active,
        Nk_mean, curv, Q_vals, Qext_vals,
        Qavg_vals
    ) = run_adaptive_phase_diagram(

        # ---------------------------------------------------------------------
        # System and Hamiltonian parameters
        # ---------------------------------------------------------------------

        # Number of complex-fermion sites.
        #
        # The Hamiltonian is written in a Majorana basis and has dimension 2L.
        # Output filenames therefore begin with 2*L.
        L=250,

        # Uniform hopping amplitude t.
        t0=0.5,

        # Uniform p-wave pairing amplitude Delta.
        #
        # The choice t0 = delta0 corresponds to the fully critical limit of
        # the clean quadratic chain in the convention used in this notebook.
        delta0=0.5,

        # Dimensionless parameter used to set the chemical potential:
        #
        #     mu0 = -int_stagg * (t0 + delta0)
        #
        # For the present values,
        #
        #     int_stagg = 1
        #
        # gives mu0 = -1.
        int_stagg=1,

        # Ratio between the two interaction couplings used by the mean-field
        # Hamiltonian:
        #
        #     UY = 2*g
        #     VY = VY_to_UY_ratio * UY
        #
        # VY_to_UY_ratio = 1 gives VY = UY and therefore uniform interaction
        # couplings in the Majorana representation.
        VY_to_UY_ratio=1.0,

        # Temperature parameter passed to the System class.
        #
        # Important: the present implementation constructs the correlation
        # matrix using the zero-temperature occupation
        #
        #     heaviside(-energy, 0.5).
        #
        # Therefore, T is currently not used to generate finite-temperature
        # Fermi occupations and does not affect the numerical result.
        T=1.0,

        # Boundary-condition switch:
        #
        # pbc = 1:
        #     periodic boundary conditions.
        #
        # pbc = 0:
        #     open boundary conditions.
        pbc=1,

        # Include the interaction-generated gamma-gamma and eta-eta mean-field
        # contributions. The random class-D terms are controlled by W_ratio.
        trs=trs,

        # Relative strength of the TRS-breaking random hopping.
        #
        # With W_ratio = 1, the same-Majorana random hopping has amplitude W.
        W_ratio=W_ratio,

        # ---------------------------------------------------------------------
        # Interaction grid
        # ---------------------------------------------------------------------

        # Values of the interaction parameter g.
        #
        # Internally, the interaction coupling is defined as
        #
        #     UY = 2*g.
        #
        # This array contains 41 uniformly spaced values over
        #
        #     -0.5 <= g <= 0.5.
        g_values=np.linspace(-0.5, 0.5, 41),

        # An extended interaction interval can instead be selected with:
        #
        # g_values=np.linspace(-0.5, 1.5, 81),

        # ---------------------------------------------------------------------
        # Disorder grid
        # ---------------------------------------------------------------------

        # Values of the overall disorder strength W.
        #
        # The quadratic spacing
        #
        #     W_i = i**2 / 400,  i = 0,...,20
        #
        # gives 21 points over 0 <= W <= 1 and provides a denser grid near
        # weak disorder.
        #
        # In class D, W sets the scale of both:
        #
        # 1. the BDI-type random terms;
        # 2. the gamma-gamma and eta-eta random terms multiplied by W_ratio.
        WX_range=np.asarray([i**2 / 400 for i in range(21)]),

        # An alternative nonuniform disorder grid is:
        #
        # WX_range=np.asarray([
        #     (2*i/20) * (1 - np.exp(-4*(2*i/20)**2))
        #     for i in range(21)
        # ]),

        # ---------------------------------------------------------------------
        # Self-consistent mean-field iteration
        # ---------------------------------------------------------------------

        # Amplitude of the random initial mean-field configuration.
        #
        # The initial array contains 6L correlators drawn uniformly from
        #
        #     [-delta_mf, delta_mf].
        delta_mf=1e-3,

        # Linear mixing or damping parameter:
        #
        #     npf_new = (1 - fric)*npf_output + fric*npf_old.
        #
        # Larger fric retains more of the previous iteration and may improve
        # stability, although it generally slows convergence.
        fric=0.5,

        # Mean-field convergence tolerance.
        #
        # A realization is marked as converged when
        #
        #     max(abs(npf_old - npf_output)) < mf_precision.
        mf_precision=1e-9,

        # Maximum number of self-consistency iterations for one disorder
        # realization at one point of the (W,g) grid.
        jj_iter=800,

        # ---------------------------------------------------------------------
        # Entanglement entropy and effective central charge
        # ---------------------------------------------------------------------

        # Smallest subsystem length included in the entropy calculation.
        lx0=10,

        # Largest subsystem length included in the entropy calculation.
        lx1=40,

        # Increment between consecutive subsystem lengths.
        #
        # With lx0=10, lx1=40, and dlx=5, the subsystem lengths are
        #
        #     lx = 10, 15, 20, 25, 30, 35, 40.
        dlx=5,

        # Step between starting positions of subsystem segments.
        #
        # Rq = 1:
        #     use every allowed segment position.
        #
        # Rq = 2:
        #     use every second allowed position.
        #
        # The entropy for a given subsystem length is averaged over the
        # selected segment positions.
        Rq=1,

        # ---------------------------------------------------------------------
        # Disorder averaging and adaptive stopping
        # ---------------------------------------------------------------------

        # Maximum number of disorder realizations evaluated at each active
        # point of the (W,g) grid.
        dis_number_max=1000,

        # Threshold for the change of the fitted effective central charge.
        #
        # After min_realizations_before_stop successful realizations, a point
        # may be deactivated when
        #
        #     abs(c_current - c_previous) < central_tol.
        central_tol=1e-3,

        # Maximum number of consecutive mean-field convergence failures
        # allowed at one grid point.
        #
        # After this number is reached, the corresponding point is removed
        # from subsequent disorder realizations.
        max_consecutive_fail=20,

        # Minimum number of successfully accumulated disorder realizations
        # before adaptive stopping is allowed.
        #
        # Since this is set equal to dis_number_max, early stopping is
        # effectively disabled before all 1000 realizations have been reached.
        min_realizations_before_stop=1000,

        # Threshold for the standard error of the fitted central charge.
        #
        # A grid point may be deactivated when
        #
        #     standard_error(c) < se_tol,
        #
        # provided that min_realizations_before_stop has been reached.
        se_tol=1e-3,

        # ---------------------------------------------------------------------
        # Parallelization
        # ---------------------------------------------------------------------

        # Number of worker processes used to evaluate active (W,g) points in
        # parallel.
        #
        # processes = None:
        #     use all CPU cores returned by multiprocessing.cpu_count().
        #
        # To reserve some CPU cores, replace None with an explicit integer,
        # for example:
        #
        #     processes=120
        #
        # on a machine with 128 available cores.
        processes=None,

        # ---------------------------------------------------------------------
        # Output
        # ---------------------------------------------------------------------

        # Directory in which the running disorder-averaged arrays are saved.
        #
        # The files are overwritten after each disorder realization, so the
        # directory contains the latest accumulated averages.
        out_dir="../mean_field_data/raw/data_fully_critical_chain_D/",

        # Intended filename prefix.
        #
        # Important: save_prefix is accepted by run_adaptive_phase_diagram(),
        # but the current saving commands construct fixed filenames and do not
        # use this parameter.
        save_prefix="_"
    )


Parallel over grid: |W|=21, |g|=41, processes=128
Up to 1000 realizations; early stop when Δc̄ < 0.001 or SE(c) < 0.001 (after 1000).
Realization 1/1000 — active points: 861
Realization 2/1000 — active points: 861
Realization 3/1000 — active points: 861
Realization 4/1000 — active points: 861
Realization 5/1000 — active points: 861
Realization 6/1000 — active points: 861
Realization 7/1000 — active points: 861
Realization 8/1000 — active points: 861
Realization 9/1000 — active points: 861
Realization 10/1000 — active points: 861
Realization 11/1000 — active points: 861


Process ForkPoolWorker-1397:
Process ForkPoolWorker-1378:
Process ForkPoolWorker-1394:
Process ForkPoolWorker-1331:
Process ForkPoolWorker-1365:
Process ForkPoolWorker-1404:
Process ForkPoolWorker-1307:
Process ForkPoolWorker-1354:
Process ForkPoolWorker-1398:
Process ForkPoolWorker-1384:
Process ForkPoolWorker-1342:
Process ForkPoolWorker-1323:
Process ForkPoolWorker-1332:
Process ForkPoolWorker-1387:
Process ForkPoolWorker-1346:
Process ForkPoolWorker-1372:
Process ForkPoolWorker-1301:
Process ForkPoolWorker-1374:
Process ForkPoolWorker-1386:
Process ForkPoolWorker-1343:
Process ForkPoolWorker-1303:
Process ForkPoolWorker-1390:
Process ForkPoolWorker-1406:
Process ForkPoolWorker-1373:
Process ForkPoolWorker-1310:
Process ForkPoolWorker-1363:
Process ForkPoolWorker-1325:
Process ForkPoolWorker-1399:
Process ForkPoolWorker-1326:
Process ForkPoolWorker-1322:
Process ForkPoolWorker-1313:
Process ForkPoolWorker-1388:
Process ForkPoolWorker-1315:
Process ForkPoolWorker-1317:
Process ForkPo

KeyboardInterrupt: 